# PhenoCycler preprocessing and quality control

Inspect the data that entered the pancreas analysis. Follow a donor from its available protein panel to native section coordinates and the expression-only UMAP. The supplied AnnData already includes acquisition, segmentation, spillover, normalization and typing QC; this notebook reads those results.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/01_preprocessing.ipynb).

## 1. Load the processed PhenoCycler data

All 25 pancreas sections have a full donor file. The separate random sample contains up to 4,000 admitted cells per donor, selected independently of expression and phenotype.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
release = validate_release(data)

import h5py
DONOR = "6347"
manifest = pd.read_csv(data / "pancreas_manifest.csv", dtype={"donor":str}).set_index("donor")
if DONOR not in manifest.index:
    raise ValueError("Choose a donor present in pancreas_manifest.csv")
random = sc.read_h5ad(data / "pancreas_random.h5ad")
assert set(random.obs.donor.astype(str)) == set(manifest.index)
print(f"{len(manifest)} pancreatic sections · {manifest.cells.sum():,} admitted cells")
display(manifest.loc[[DONOR],["disease","cells","markers","usable_markers","umap_sample","finite_umap"]])

## 2. Which proteins and control channels were available?

Each row represents one acquired section. The figure includes the hormone, epithelial, vascular, lymphatic, neural and immune markers used in the phenotype profiles alongside control channels and other measurements that varied by section.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
core_markers = list(dict.fromkeys(PAN_STRUCTURE + IMMUNE))
available = {}
for donor_id, row in manifest.iterrows():
    # Read only the feature annotations, not an entire multi-million-cell expression matrix.
    with h5py.File(data / row.path, "r") as f:
        var = ad.io.read_elem(f["var"])
    available[donor_id] = var.available
    assert int(var.available.sum()) == int(row.usable_markers)
order = sorted(manifest.index, key=lambda d: (["ND","AAb+","T1D"].index(manifest.at[d,"disease"]), int(d)))
all_markers = pd.DataFrame(available).T.loc[order].fillna(False)
detected = all_markers.sum(axis=0).sort_values()
variable = [key for key in detected.index if 0 < detected[key] < len(order) and key not in core_markers]
markers = list(dict.fromkeys(core_markers + variable))
avail = all_markers.reindex(columns=markers, fill_value=False)
fig, ax = plt.subplots(figsize=(max(8.8,.45*len(markers)),8.1))
ax.imshow(avail.to_numpy(dtype=int),aspect="auto",cmap=ListedColormap(["#e4e8eb","#326e83"]),vmin=0,vmax=1,interpolation="none")
ax.set(xticks=np.arange(len(markers)),xticklabels=markers,yticks=np.arange(len(order)),
       yticklabels=[d+"  ·  "+manifest.at[d,"disease"] for d in order], title="Protein and control-channel availability in pancreatic sections")
ax.tick_params(axis="x",labelrotation=48)
for label in ax.get_xticklabels():
    label.set_ha("right"); label.set_rotation_mode("anchor")
fig.legend(handles=[Patch(color="#326e83",label="Available"),Patch(color="#e4e8eb",label="Unavailable")],
           loc="upper center",bbox_to_anchor=(.53,.995),ncol=2,frameon=False)
fig.subplots_adjust(left=.16,bottom=.22,top=.88)
plt.show()

## 3. Cell-level measurement and spatial QC

For the selected donor, map the number of finite protein measurements in the random sample back to the tissue. A measured zero counts as measured. Fragment-overlapping hormone measurements were masked by the original typing QC.

In [ ]:
section = random[random.obs.donor.astype(str).eq(DONOR)].copy()
scaled = np.asarray(section.layers["typing_scaled"])
section.obs["Measured proteins"] = np.isfinite(scaled).sum(axis=1)
if not section.obs.qc_analysis_eligible.fillna(False).all():
    raise ValueError("The admitted sample includes an ineligible cell")
fig, axes = plt.subplots(1,2,figsize=(10.8,4.5),gridspec_kw={"width_ratios":[1.12,1]})
sc.pl.embedding(section,basis="spatial",color="Measured proteins",cmap="viridis",size=8,
                ax=axes[0],frameon=False,show=False,title=f"Native section  ·  donor {DONOR}")
axes[0].set_aspect("equal"); axes[0].invert_yaxis()
v = section.obs["Measured proteins"].value_counts().sort_index()
axes[1].bar(v.index,v.values,color="#387c90",width=.8)
axes[1].set(xlabel="Proteins with a finite value",ylabel="Sampled cells",title=f"{section.n_obs:,} random cells")
fig.tight_layout()
plt.show()
display(pd.DataFrame({"selected donor":[DONOR],"available proteins":[int(manifest.at[DONOR,"usable_markers"])],
                      "cells with masked hormone measurements":[int(section.obs.hormone_fragment_masked.sum())],
                      "sampled cells":[section.n_obs]}))

## 4. Which cells appear on the shared UMAP?

This UMAP uses a separate display sample enriched for rare phenotypes. The table compares the number selected with the number that had a complete common-marker profile and received coordinates. Whole-donor abundance uses the full donor files.

In [ ]:
atlas = sc.read_h5ad(data / "pancreas_umap.h5ad")
finite = np.isfinite(atlas.obsm["X_umap"]).all(axis=1)
coverage = pd.crosstab(atlas.obs.donor.astype(str), finite).reindex(index=manifest.index, columns=[False,True],fill_value=0)
assert coverage[True].eq(manifest.finite_umap).all()
coverage.columns = ["No coordinates","With coordinates"]
display(coverage.join(manifest[["disease","cells"]]).rename(columns={"cells":"Admitted in full section"}))
print("Proceed to 02_phenotyping.ipynb for shared expression maps and marker profiles.")